In [0]:
%sql

CREATE OR REPLACE TABLE medical_data.gold.kpi_encounter AS
WITH class_counts AS (
    SELECT 
        f.quarter,
        d.encounter_class,
        COUNT(*) AS class_count
    FROM medical_data.gold.fact_encounters f
    JOIN medical_data.gold.dim_encounter d
        ON f.encounter_id = d.encounter_id
    GROUP BY f.quarter, d.encounter_class
),
total_counts AS (
    SELECT 
        quarter,
        COUNT(*) AS total_count
    FROM medical_data.gold.fact_encounters
    GROUP BY quarter
)
SELECT 
    c.quarter,
    c.encounter_class,
    c.class_count,
    t.total_count,
    ROUND((c.class_count * 100.0) / t.total_count, 2) AS percentage
FROM class_counts c
JOIN total_counts t
    ON c.quarter = t.quarter
ORDER BY c.quarter, percentage DESC;

In [0]:
%sql
select * from medical_data.gold.kpi_encounter;

In [0]:
%sql

CREATE OR REPLACE TABLE medical_data.gold.kpi_encounter_duration AS

SELECT 
    month,
    CASE 
        WHEN time_diff > 24 THEN 'Above_24_Hours'
        ELSE 'Below_24_Hours'
    END AS duration_category,
    COUNT(*) AS cnt,
    
    ROUND(
        COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (PARTITION BY month),
        2
    ) AS percentage
FROM medical_data.gold.fact_encounters
GROUP BY 
    month,
    CASE 
        WHEN time_diff > 24 THEN 'Above_24_Hours'
        ELSE 'Below_24_Hours'
    END
ORDER BY month, percentage DESC;

In [0]:
%sql

CREATE OR REPLACE TABLE medical_data.gold.kpi_zero_coverage AS

SELECT 
    month,
    payer_id,
    COUNT(*) AS total_encounters,
    SUM(CASE WHEN payer_coverage = 0 THEN 1 ELSE 0 END) AS zero_coverage,
    ROUND(
        SUM(CASE WHEN payer_coverage = 0 THEN 1 ELSE 0 END) * 100.0 / COUNT(*),
        2
    ) AS zero_coverage_percentage,
    ROUND(
        SUM(CASE WHEN payer_coverage > 0 THEN 1 ELSE 0 END) * 100.0 / COUNT(*),
        2
    ) AS covered_percentage
FROM medical_data.gold.fact_encounters
GROUP BY month, payer_id;

In [0]:
%sql

CREATE OR REPLACE TABLE medical_data.gold.kpi_top_procedures AS

SELECT 
    half_yearly,
    code AS procedure_code,
    description,
    COUNT(*) AS procedure_count,
    ROUND(AVG(base_cost), 2) AS avg_base_cost

    -- need to change the base_cost to integer
FROM medical_data.silver.procedures_s
GROUP BY half_yearly, code, description
ORDER BY avg_base_cost DESC
LIMIT 10;

In [0]:
%sql

CREATE OR REPLACE TABLE medical_data.gold.kpi_avg_claim_cost AS

SELECT 
    payer_id,

    COUNT(*) AS total_encounters,

    ROUND(AVG(total_claim_cost), 2) AS avg_claim_cost

FROM medical_data.gold.fact_encounters

GROUP BY payer_id

ORDER BY avg_claim_cost DESC;

In [0]:
%sql

CREATE OR REPLACE TABLE medical_data.gold.kpi_readmission AS

WITH ordered AS (
    SELECT 
        patient_id,
        encounter_id,
        start AS encounter_start,
        LAG(start) OVER (
            PARTITION BY patient_id 
            ORDER BY start
        ) AS prev_start
    FROM medical_data.gold.fact_encounters
),

readmission_flag AS (
    SELECT 
        patient_id,
        encounter_id,
        encounter_start,
        CASE 
            WHEN prev_start IS NOT NULL 
                 AND DATEDIFF(encounter_start, prev_start) <= 30
            THEN 1
            ELSE 0
        END AS is_readmission
    FROM ordered
)

SELECT 
    DATE_FORMAT(encounter_start, 'yyyy-MM') AS month,

    COUNT(*) AS total_encounters,

    SUM(is_readmission) AS readmissions,

    ROUND(
        SUM(is_readmission) * 100.0 / COUNT(*),
        2
    ) AS readmission_rate
FROM readmission_flag
GROUP BY DATE_FORMAT(encounter_start, 'yyyy-MM')
ORDER BY month;